# Numerical Differentiation

Chapter 1 of the TN23015 book: its explanation and its exercises. Run the first cell before anything else, then work down the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tn23015 import check_answers, given, load_data, md, print, row, show, slider

In this notebook, you will explore taking numerical derivatives of functions and of sampled data, and how their accuracy depends on the step size.

> **Learning goals**
>
> 1. Write a function to calculate the numerical derivative of an arbitrary function using both the forward difference and the central difference approximation
> 2. Interpret the error made in central and forward numerical derivatives
> 3. Calculate the numerical derivative of a dataset

## Finite difference methods for calculating derivatives of functions

### Introduction

We will begin with considering calculating the numerical derivatives of functions, specifically using the [Finite difference](https://en.wikipedia.org/wiki/Finite_difference) technique.

In the simplest case, the derivative we want to calculate might be the derivative of a functions that we have an analytical expression for, such as $f(x) = ax^2$. 
In contrast to analytical integration, which can sometimes be challenging to perform and find a closed solution for, taking derivatives of functions is in general not so difficult (you likely learned this in your first calculus course!). 
For example, the derivative of $f(x)$ above is simply $f'(x) = 2ax$. 

You might ask: if it is always possible to analytically calculate the derivative of a function, why would I bother calculating the derivative with a computer? Some quick answers:

* It is handy 
* Sometimes you don't know the function
* Finite difference is a key concept in solving many numerical problems

A first reason is that it is just handy sometimes to calculate the derivative numerically, it saves you time in having to calculate the analytical derivative by hand. 
For example, later, when we solve differential equations, we will use code routines from libraries to get solutions, and then it is handy for us if we only need to provide the function, and not its derivatives. 
The same is true for curve fitting: curve fitting routines also use derivatives of the fitting functions, which they calculate numerically so that the user using them doesn't have to think about it. 

A second reason is that in a computer program, you can implement functions that are not necessarily simple analytical formulas. 
For example, $f(x)$ in a computer could return the result of a numerical simulation. 
Or it could be a random number. 
Or it could be a piecewise function:

```
def my_funny_function(x):
    if (x<0):
        return 0
    else:
        return x**2
```

If we calculate the derivative of `my_funny_function(x)` numerically, then I don't even need to know what happens inside of the function.

Finally, a third important reason is that the concept of finite difference methods for calculating derivatives also forms the basis of important numerical algorithms used for solving equations in physics, such as Maxwell's equations, the Schroedinger equation, and more. 

### Overview of finite difference methods

All finite difference methods are based on approximating a derivative of a function at a point by the slope of a line connecting two points separated by a distance $h$:

$$
\frac{df}{dx} \approx 
\frac{f(x+h) - f(x)}{h}
$$

The idea is illustrated in this image from wikipedia, where the formula above calculates the *secant*: 

![](https://raw.githubusercontent.com/curiousbeams/lecture-tn23015-computational-science/main/figures/541px-Derivative.svg.png)

The approximation becomes exact in the limit $h \rightarrow 0$, but in a computer we will work with a non-zero $h$.
A subtle point of the above equation is: to which point $x$ do we assign the value of $df/dx$ from this formula? 
Do we choose it to be the value of the derivative at $x$? Or the value at $x+h$? Or somewhere in between?

Generally speaking, people distinguish three different common choices for this: 

* **Backward difference** Approximates that the secant is the derivative in the right side of the plot $df/dx = f'(x+h)$
* **Forward difference** Approximates that the secant is the derivative at the left side of the above plot: $df/dx = f'(x)$
* **Center difference** Approximates that the secant is the derivative in the middle of the plot $df/dx = f'(x+h/2)$

In the specific example above, we can see that the secant will over-estimate the derivative at position $x$ (the secant is steeper than the slope of the function at that point): this means that the forward difference approximation will overestimate the derivative in this case. 
Conversely, in the example above, the secant will underestimate the slope of the function at $x+h$, and so the backwards difference will underestimate the derivative in this case. 

To explain the terminology further, it is useful to remember that the forward difference uses a point in *front* of where we are (at larger x) to estimate the derivative, the backwards derivative uses a point *behind* where we are to estimate the derivative, and the center difference uses equally spaced points forward and backwards to estimate the derivative. 
These three are illustrated here for estimating the derivative at a position $x_0$:

![](https://raw.githubusercontent.com/curiousbeams/lecture-tn23015-computational-science/main/figures/finite_difference.png)

And, as you can already maybe guess looking at these two examples, the center difference gives the most accurate approximation of the derivative $df/dx$ at position $x_0$, something we will explore below.

### Definition of the three techniques

Having explored the basic ideas above, we give here the formal definitions of the three techniques. The forward difference as:

$$
f'_F(x,h) = \frac{f(x+h) - f(x)}{h}
$$

The backward difference is defined as:

$$
f'_B(x,h) = \frac{f(x) - f(x-h)}{h}
$$

and the center difference as:

$$
f'_C(x,h) = \frac{f(x+h/2) - f(x-h/2)}{h}
$$

### Order of the errors made

Using a Taylor expansion, one can also estimate the errors made in the above approximations of the derivative (see [Relation with derivatives](https://en.wikipedia.org/wiki/Finite_difference#Relation_with_derivatives)).

The forward difference approximation results in an error $\epsilon_F = f'_F(x,h) - f'(x)$ that is *first order* in $h$, ie. proportional to $h$:

$$
\epsilon_F \propto h |f''(x)| \hspace{1em} {\rm as}\  h \rightarrow 0
$$

where $f''(x)$ is the second derivative of the function. 

A common notation for this, borrowed from computer science, is big-O notation, which is used to define the asymptotic limits of functions.
Basically, $\epsilon = \mathcal{O}(h)$ means the same thing as $\epsilon \propto h$, and in words we would read this as "the error is order h".
We can then write:

$$
\epsilon_F = \mathcal{O}(h) \hspace{1em} {\rm as}\  h \rightarrow 0
$$

Similarly, the error for the backwards difference is also $\mathcal O(h)$:

$$
\epsilon_B = \mathcal{O}(h) \hspace{1em} {\rm as}\  h \rightarrow 0
$$

The center difference is actually more accurate:

$$
\epsilon_C = \mathcal{O}(h^2) \hspace{1em} {\rm as}\  h \rightarrow 0
$$

It is often used for this reason, although it can suffer from problems with periodic functions.

### Higher order approximations of derivatives 

If one would like to have a more accurate estimate of the derivative without taking a smaller value of $h$, one can also use higher order approximations of the derivative: for example, the following forward difference approximation is a more complicated formula: 

$$
\frac{df}{dx} \approx \frac{-f(x+2h)+4f(x+h)-3f(x)}{2h}
$$

but gives an error order $\mathcal O(h^2)$. 
This forms the basis of the Runge-Kutta techniques for solving differential equations that we will look at later in the course.

### Higher order derivatives

One can also extend the above techniques to calculate second-order derivatives (or even higher if needed). 
For example, in the center difference approximation,  

$$
\frac{d^2f}{dx^2} \approx \frac{f(x+h)-2f(x)+f(x-h)}{h^2}
$$

which is a formula we will use later when we are solving the Poisson equation. 

## Benchmarking the numerical derivative

To illustrate the above concepts, we will use python code to calculate the derivative of the function: 

$$
f(x) = 1+\frac{1}{2} \tanh 2x
$$

We will do this using different techniques, and perform "benchmarking" of these techniques by comparing the numerically calculated derivative with the analytical answer.

### Exercise 1.1

Calculate the derivative of this function analytically (ie. with a pen and paper). Program the two results into two python functions.
Create variable arrays `y` and `y_prime` with the value $y = f(x)$ and $y' = f'(x)$ respectively, with x in the range $-2 \leq x \leq 2$ and 100 points.
Make plots of $f(x)$ and its derivative $f'(x)$.

In [ ]:
# Fill these in as you work through the exercise.
_ax = x = y = y_prime = None

def f(x):
    return None

def fp(x):
    return None

# x = ....
# y = ....
# y_prime = ....

# Now the plot (with labels and legend of course!)
# _fig, _ax = plt.subplots()
# ...

# Do not edit or remove the boiler-plate code below.
show(_ax, x=x, y=y, y_prime=y_prime, f=f, fp=fp)

In [ ]:
check_answers(x=x, y=y, y_prime=y_prime, key="answer_3_01")

<details>
<summary><strong>Self check</strong></summary>

* Does your plot have axis labels?
* Does it have a legend?

</details>

### Exercise 1.2

Now, we will write a function to calculate the derivative numerically with a given step size $h$ and compare our numerically calculated derivative with that of the analytical answer.

Write two functions that calculate the derivative of a function numerically, one using the forward difference method, and one using the central difference method.

Use these functions to calculate $f'(x)$ at $x=1$ for $h=0.5$, and compare your numerically calculated derivatives to the correct answer from the analytical formula.

In [ ]:
# Fill these in as you work through the exercise.
error_center = error_forward = yd_analytical = yd_center = yd_forward = None

def diff_forward(f,x,h):
    return None

def diff_center(f,x,h):
    return None

x0=1
h=0.5

# yd_analytical = 
# yd_forward = 
# yd_center = 
# error_forward = 
# error_center = 

# Do not edit or remove the boiler-plate code below.
if given(
    error_center,
    error_forward,
    yd_analytical,
    yd_center,
    yd_forward,
    diff_forward,
    diff_center,
):
    print("Forward:    %f   Error: %f" % (yd_forward, error_forward))
    print("Central:    %f   Error: %f" % (yd_center, error_center))
    print("Analytical: %f" % yd_analytical)

show(
    error_center=error_center,
    error_forward=error_forward,
    yd_analytical=yd_analytical,
    yd_center=yd_center,
    yd_forward=yd_forward,
    diff_forward=diff_forward,
    diff_center=diff_center,
)

In [ ]:
check_answers(
    forward=(yd_forward, error_forward),
    center=(yd_center, error_center),
    yd_analytical=yd_analytical,
    key="answer_3_02",
)

<details>
<summary><strong>Self check</strong></summary>

Which is more accurate, the forward or the central derivative?

</details>

### Exercise 1.3

Repeat the calculation of Exercise 1.2 but now at $x=0$.

In [ ]:
# Fill these in as you work through the exercise.
error_center_2 = error_forward_2 = yd_analytical_2 = yd_center_2 = yd_forward_2 = None

# yd_analytical_2 = 
# yd_forward_2 = 
# yd_center_2 = 
# error_forward_2 = 
# error_center_2 = 

# Do not edit or remove the boiler-plate code below.
if given(error_center_2, error_forward_2, yd_analytical_2, yd_center_2, yd_forward_2):
    print("Forward:    %f   Error: %f" % (yd_forward_2, error_forward_2))
    print("Central:    %f   Error: %f" % (yd_center_2, error_center_2))
    print("Analytical: %f" % yd_analytical_2)

show(
    error_center=error_center_2,
    error_forward=error_forward_2,
    yd_analytical=yd_analytical_2,
    yd_center=yd_center_2,
    yd_forward=yd_forward_2,
)

In [ ]:
check_answers(
    forward=(yd_forward_2, error_forward_2),
    center=(yd_center_2, error_center_2),
    yd_analytical_2=yd_analytical_2,
    key="answer_3_03",
)

### Exercise 1.4

Calculate the derivative of the function for $x$ ranging from $-2 \leq x \leq 2$ with 100 points using the forward and the central difference methods with $h = 0.5$. 

You can make use of the functions you defined above.

Make a plot of the forward difference derivative and the central difference derivative vs. x, along with the analytical solution.

In [ ]:
# Fill these in as you work through the exercise.
_ax = x_2 = yd_analytical_3 = yd_center_3 = yd_forward_3 = None

# The spacing
h_2=0.5

# Our array of points at which to perform the calculations
# x_2=np.linspace(__,__,__)

# The derivatives
# yd_center_3 = ___
# yd_forward_3 = ___
# yd_analytical_3 = ____

# The plot
# _fig, _ax = plt.subplots()
# ...

# Do not edit or remove the boiler-plate code below.
show(_ax, x=x_2, yd_analytical=yd_analytical_3, yd_center=yd_center_3, yd_forward=yd_forward_3)

In [ ]:
check_answers(
    x_2=x_2,
    yd_center_3=yd_center_3,
    yd_forward_3=yd_forward_3,
    yd_analytical_3=yd_analytical_3,
    key="answer_3_04",
)

<details>
<summary><strong>Self check</strong></summary>

* Does your plot have axis labels?
* Does it have a legend?

What do you see in the plot? What can you interpret in the plot?

</details>

### Exercise 1.5

Here, you will make a plot of the error of the forward difference derivative, and also a plot of the function f(x), in two plots beside each other.
The idea is to try to understand the value of the error (is it large, is it small, is it positive, is it negative) based on the shape of the function.

Your x-range should extend from -2 to 2 with 100 points. 

To make two plots beside each other, you will use the matplotlib subplot capabilities:

Start your subplots using the command `_fig, _axs = plt.subplots(1, 2, figsize=(12,4))`.
To make your left hand plot, run the command `_ax = _axs[0]`, then use the usual plot commands to make your plot. 
To make the right hand plot, use the command `_ax = _axs[1]` and then add the commands for the second plot.

Your left plot should plot the error, defined as the difference between the derivative calculated by the finite difference method, and the analytical answer:

$$
\mathrm{error} = 
\left (\frac{df}{dx} \right)_{\mathrm{fwd}} - 
\left ( \frac{df}{dx} \right)_{\mathrm{analytical}}
$$

In this plot of the error, make a vertical line at $x=0$ using the  `_ax.axvline()` command, and a horizontal line at $y=0$ using the `_ax.axhline()` command. 

In the right plot, plot the function $f(x)$ as a function of $x$. For this plot, add a vertical line at $x=0$.

In [ ]:
# Fill this in as you work through the exercise.
_fig = None

# _fig, _axs = plt.subplots(1, 2, figsize=(12,4))
# _ax = _axs[0]

# _ax.plot(....)
# ....

# _ax.axvline(0, ls=":", color="grey")
# _ax.axhline(0, ls=":", color="grey")

# _ax = _axs[1]
# _ax.plot(....)
# ....

#_ax.axvline(0, ls=":", color="grey")

# Do not edit or remove the boiler-plate code below.
show(_fig)

<details>
<summary><strong>Self check</strong></summary>

* Does your plot have axis labels?

Can you explain why the forward difference overestimates the derivative for $x = -0.5$ and underestimates it for $x = 0.5$?

> **Hint**
>
> Take a look at the shape of the function as you plotted it in Exercise 1.1, think about how the forward difference calculates the derivative, and then think about how the shape of the function affects the error in the forward derivative calculation.

</details>

## Scaling of error with step size

In this section, we will calculate the scaling of the error incurred by different techniques for numerical differentiation, as a function of the step size $h$, and compare this with what we expect.

### Exercise 1.6

Write a loop that calculates the numerical derivative $f'(x)$ at $x=1$, using the forward, center, and backwards difference formulas.
In your loop, perform your calculation for 100 different values of $h$ evenly spaced on a log scale from $h = 1\times10^{-6}$ to $h = 1$.
For this, you can use the [`geomspace`](https://numpy.org/doc/stable/reference/generated/numpy.geomspace.html) function.

You can also use [`logspace`](https://numpy.org/doc/stable/reference/generated/numpy.logspace.html) if you want, but it is a bit trickier, since `logspace` takes the exponents of the endpoints rather than the endpoints themselves. 

Make a log-log plot of the error $\epsilon$ vs $h$ for the three techniques (three lines on the same graph), with $\epsilon$ defined as:

$$
\epsilon = |f'_{calc}(1) - f'(1)|
$$

where $f'_{calc}$ is the numerically calculated derivative and $f'$ is the analytical answer.

In [ ]:
# Fill these in as you work through the exercise.
_ax = err_backward = err_center = err_forward = h_3 = None

N = 100

# Code to define any functions you need, calculate the derivatives
# for 100 values of h, and make a plot with axis labels.
# h_3 = ...

# Your code should produce three numpy arrays:
# err_forward, err_center, err_backward, along with the array h
# of points spaced equally on a logarithmic axis

def diff_backward(f,x,h):
    return None

# err_forward = ...
# err_center = ...
# err_backward = ...

# It's handy on this plot to have a grid
# _fig, _ax = plt.subplots()
# _ax.grid()

# Do not edit or remove the boiler-plate code below.
show(
    _ax,
    err_backward=err_backward,
    err_center=err_center,
    err_forward=err_forward,
    h=h_3,
    diff_backward=diff_backward,
)

In [ ]:
check_answers(
    h_3=h_3,
    err_forward=err_forward,
    err_center=err_center,
    err_backward=err_backward,
    key="answer_3_06",
)

<details>
<summary><strong>Self check</strong></summary>

* Does your plot have: legend, axis labels? 
* Do the different techniques show the scaling that you would expect? 
* What happens to the center difference curve for the smallest values of h?

</details>

Once you have made your own plot, compare it with the one below, where you can also move the point $x_0$ and see the three secants themselves.
Shrinking $h$ reduces the truncation error, until the round-off error in $f(x+h) - f(x)$, which grows like $\epsilon_\mathrm{mach}/h$ with $\epsilon_\mathrm{mach}$ the machine precision, takes over.
Switch to 32-bit precision to see how much sooner that happens when the numbers carry fewer digits.

In [ ]:
# An interactive figure from the book: run this cell to draw it. It loads from the internet.
from tn23015 import interactive_figure

await interactive_figure(
    "https://curiousbeams.github.io/numerics-widgets/notebooks/finite-difference-error.html",
    cells=["controlsView", "secantView", "errorView", "readoutView"],
    # theme="dark",  # if the colours look wrong, uncomment this (or use "light")
)

## Numerical derivatives of data

Here, we will explore various aspects of taking the derivative of numerically sampled datasets, for example like the data that might come out of a measurement in a physics experiment.

In this case, things are a bit simpler, as we do not need to think about the value of the spacing $h$ since this is determined by the spacing in the sampling of the data.
Let's look at an explicit example. 
Say we have a data set of 5 points corresponding to voltages measured at 5 different times:

```
# t (s), V (V)
0, 1
1, 2
2, 5
3, 10
4, 17
```

If we loaded this into python, we would then get two vectors, one for the time, and one for the voltage:

```
t = [0, 1, 2, 3, 4]
v = [1, 2, 5, 10, 17]
```

We can then define the derivative $v'(t) = dV/dt$ as:

$$
v'(t_n) = \frac{v(t_{n+1}) - v(t_n)}{t_{n+1} - t_n}
$$

In the example above, the measurements are evenly spaced in time so $t_{n+1} - t_{n} = \Delta t = 1$ for all $n$, and then we have a simple result for the derivative array:

```
v_prime = [1, 3, 5, 7]
```

Note that when we take the derivative of the data, we always end up with one less point in the resulting array than we started with!
This is because there is no $n+1$ point for the last point in the array `v` that we started with. 

Using the finite difference method, it is pretty clear what the array `v_prime` should be. 
But at which time do these points correspond to?
Clearly we cannot simply reuse the original array `t` since it is a different length.

The answer is that it depends on which approximation we use (forward, center, or backward difference).
In fact, for numerical data, the `v_prime` array is the same for all three cases! 
The difference is which *times* we assign these values to. 
If we take the forward difference approximation, we would take:

```
t_forward = [0, 1, 2, 3]
```

For the backward difference approximation, we would take:

```
t_backward = [1, 2, 3, 4]
```

while for the center difference, we could take:

```
t_center = [0.5, 1.5, 2.5, 3.5]
```

As a final note, when working with numerical data, we can also define a center difference that takes the distance for the derivative as twice the time separation of the points, $h = 2\Delta t$ and uses:

$$
v'(t_n) = \frac{v(t_{n+1}) - v(t_{n-1})}{t_{n+1} - t_{n-1}}
$$

In this case, our `v_prime` array would be even smaller, since there is also no point $n-1$ for the first point.
In the example above, we would then have: 

```
v_prime2 = [2, 4, 6]
```

and 

```
t_center2 = [1, 2, 3]
```

Although we lose points at both ends, this does have the advantage that there are accurate estimates of the derivatives at time values that are the same as in the original `t` array (unlike the alternative center difference above that gives accurate slopes halfway between the times in the original `t` array). 

### A sampled sine wave

We will consider a data set taken from a sinusoidal function of time $f(t) = \sin(t)$, with a relatively low sampling rate. 

For example, this could represent a measurement of an ideal sine wave that is sampled by an oscilloscope.

### Exercise 1.7

Plot the data of a sparsely sampled sinusoid with 25 points between $t = 0$ and $t = 4 \pi$.

In [ ]:
# Fill these in as you work through the exercise.
data = t = None

N_2=25
# t = np.linspace(____)
# data = ____

# Do not edit or remove the boiler-plate code below.
show(data=data, t=t)

In [ ]:
check_answers(t=t, data=data, key="answer_3_07")

### Exercise 1.8

Calculate and plot the forward derivative of the numerical data, along with the correct value from the analytical derivative of a sine function.

In [ ]:
# Fill these in as you work through the exercise.
analytical_derv = fwd_derv = t_fwd = None

# fwd_derv = np.zeros(_____)

# for i in range(_____):
#     .....

# We also need to adjust the t vector (do you remember why and how?)   
# t_fwd = ....

# analytical_derv = _____

# Do not edit or remove the boiler-plate code below.
_ax = None
if given(analytical_derv, fwd_derv, t_fwd, t):

    # And now a nice plot (for free this time)
    _fig, _ax = plt.subplots()
    _ax.plot(t, analytical_derv, '-o', label = r"$\cos(x)$")
    _ax.plot(t_fwd, fwd_derv, '-o', label = "Fwd")
    _ax.legend()
    _ax.set_xlabel('t')
    _ax.set_ylabel("f'(t)")
    _ax.axhline(0,c='grey', ls=':')

show(_ax, analytical_derv=analytical_derv, fwd_derv=fwd_derv, t_fwd=t_fwd, t=t)

In [ ]:
check_answers(fwd_derv=fwd_derv, t_fwd=t_fwd, analytical_derv=analytical_derv, key="answer_3_08")

Compare the analytical answer with the forward derivative approximation.
How accurate is it? What seems wrong with it?

### Exercise 1.9

Although the derivative seems on first inspection quite inaccurate, there is actually a small fix that can change this.
In particular, although the forward derivative calculates the derivative at the sampled time steps relatively inaccurately, it does do a pretty good job at calculating the derivative at times halfway *between* the sampled time points. 

Write code below so that you calculate the center difference derivative at points halfway between the sampling points.
Hint: you don't have to do much work!
But you will need a new time array.

Calculate the analytical answer $\cos(t)$ for the derivative at these same timesteps, and make a plot showing the two.

In [ ]:
# Fill these in as you work through the exercise.
analytical_derv_new = t_new = None

# t_new = ...
# analytical_derv_new = ...

# Do not edit or remove the boiler-plate code below.
_ax = None
if given(analytical_derv_new, t_new, fwd_derv):

    # now make the plot
    _fig, _ax = plt.subplots()
    _ax.plot(t_new, fwd_derv, '-o', label='Fwd Shifted')
    _ax.plot(t_new, analytical_derv_new, '-o', label='Analytical')
    _ax.legend()
    _ax.set_ylabel("f'(t)")
    _ax.set_xlabel('t')
    _ax.axhline(0,c='grey', ls=':')

show(_ax, analytical_derv_new=analytical_derv_new, t_new=t_new, fwd_derv=fwd_derv)

In [ ]:
check_answers(t_new=t_new, analytical_derv_new=analytical_derv_new, key="answer_3_09")

## Vectorising numerical derivatives

When calculating the numerical derivatives of data above, the simplest way to do this is to use a `for` loop to create a new array based on looping over the contents of the old array. 

As discussed in the introduction course notebooks, `for` loops in python are very slow because they are based on "interpreted loops".
A much faster way to do such calculations is to make use of "slicing" to perform "vectorised" calculations, in which the for-loops are performed by highly efficient pre-compiled libraries. 

As an example, let's consider calculating the forward derivative of a dataset stored in an array `v[i]` with `N` points as a function of the independent variable `t[i]`, which could represent a measured voltage as a function of time.

In this case, the entries of the output array `dvdt` would be given by:

```
dvdt[i] = (v[i+1]-v[i])/(t[i+1]-t[i])
```

Important to notice is that the array `dvdt` will be one point shorter than the original array `v` since that calculation can only proceed until `i=(N-1)-1` (the extra `-1` in the brackets is because python counts from zero...). 

Now, let's say we wanted to "vectorise" the calculation above.
How would we go about that? 

One way we could do that is to make a couple of new arrays, which I will do here with new names that will look a lot like the formula above using slicing:

```
v_i = v[:-1]
v_i_plus_1 = v[1:]
t_i = t[:-1]
t_i_plus_1 = t[1:]
```

Note that all of these arrays are now size `N-1`, and the first entry `v_i[0] = v[0]` while the first entry `v_i_plus_1[0] = v[1]`.
Also note the careful use of slicing to use `[:-1]` to skip the last entry and using `[1:]` to skip the first entry (remember again python counts from zero...).
By doing this, I can then use the vectorisation of numpy to calculate the output array `dvdt` in one step with no `for` loop:

```
dvdt = (v_i_plus_1 - v_i)/(t_i_plus_1 - t_i)
```

Note that this will be **much** faster since the calculation will be performed by "precompiled" for loops. 

Let's look at an example, now skipping the intermediate variable names and putting the vectorisation directly into the code:

In [ ]:
t_2 = np.linspace(0,10,1000) 
v = t_2**2
dvdt = (v[1:]-v[:-1])/(t_2[1:]-t_2[:-1])

_fig, _axs = plt.subplots(1, 2, figsize=(14,4))
_ax = _axs[0]
_ax.plot(t_2,v)
_ax.set_xlabel("t")
_ax.set_ylabel("v")
_ax = _axs[1]

# Note that we need to pick out only the first N-1 points of the t-array for plotting
# Otherwise, t and dvdt are not the same size!
_ax.plot(t_2[:-1], dvdt)
_ax.set_xlabel("t")
_ax.set_ylabel("dvdt")
show(_fig)

### Exercise 1.10

Write code that uses slicing to calculate the center derivative of a dataset constructed from the function $v = \sin(t)$ for an array `t` that runs from 0 to 10 with 1000 points. 
Evaluate the center derivative at the positions of original points, not between them, which means that your output array should be two points smaller in size than the input.

In [ ]:
# Fill these in as you work through the exercise.
dvdt_2 = None

# dvdt_2 = ...

# Have a look at the plots to see if your answer looks reasonable
_fig, _axs = plt.subplots(1, 2, figsize=(14,4))
_ax = _axs[0]
_ax.plot(t_2,v)
_ax.set_xlabel("t")
_ax.set_ylabel("v")
_ax.axhline(0,ls=":", c='gray')
_ax.axvline(0,ls=":", c='gray')

# Do not edit or remove the boiler-plate code below.
if given(dvdt_2):
    _ax = _axs[1]

    # Note that with the instructions above, this is the correct time array
    # for plotting the center derivative
    _ax.plot(t_2[1:-1], dvdt_2)
    _ax.set_xlabel("t")
    _ax.set_ylabel("dvdt")
    _ax.axhline(0,ls=":", c='gray')
    _ax.axvline(0,ls=":", c='gray')

show(_fig, dvdt=dvdt_2)

In [ ]:
check_answers(dvdt_2=dvdt_2, key="answer_3_10")

### Numpy functions for derivatives

Now that we have implemented the code for calculating derivatives ourselves, and we understand how it works, it is also useful to know that the numpy library also provides a function for calculating the differences between neighbouring elements of (multidimensional) numpy arrays, [diff](https://numpy.org/doc/stable/reference/generated/numpy.diff.html), from which a derivative follows by dividing by the spacing.

```
numpy.diff(a, n=1, axis=-1)
```
The most important parameters of this function are:

* `a`: the array you want to take the differences of
* `n`: how many times to take the difference (n=1 for a first derivative, n=2 for a second)
* `axis`: This is used for multidimensional arrays: it tells you along which "direction" the derivative should be taken. 

Here below you can find a few examples of using it:

In [ ]:
# Derivatives of a vector (1D)

x_3 = np.linspace(-5,5,100)
y_2 = np.exp(-x_3**2)

_fig, _ax = plt.subplots()
_ax.plot(x_3,y_2, label="y")

# Note that np.diff(x) is one point shorter! 
# By choosing to plot against t[0:-1], we are picking the "forward difference" approximation
_ax.plot(x_3[0:-1], np.diff(y_2), label="dy/dx")

# A second derivative
_ax.plot(x_3[0:-2], np.diff(y_2, n=2), label="d$^2$y/dx$^2$")

_ax.legend()
_ax.set_xlabel("x")
show(_ax)

In [ ]:
# Derivatives of a 2D function

x_4 = np.linspace(-5,5,100)
y_3 = np.linspace(-5,5,100)

X,Y = np.meshgrid(x_4,y_3)
Z = np.exp(-X**2-Y**2)

_fig, _axs = plt.subplots(1, 3, figsize=(14,3.5))

_ax = _axs[0]
_im1 = _ax.imshow(
    Z,
    aspect='auto',
    cmap="RdBu_r", 
    origin='lower',
    extent=(x_4[0], x_4[-1], y_3[0], y_3[-1])
)

_im1.set_clim(-1,1)
_fig.colorbar(_im1).set_label("Z")
_ax.set_xlabel("x")
_ax.set_ylabel("y")

_ax = _axs[1]
_im2 = _ax.imshow(
    np.diff(Z,axis=0),
    aspect='auto',
    cmap="RdBu_r", 
    origin='lower',
    extent=(x_4[0], x_4[-1], y_3[0], y_3[-1])
)
_fig.colorbar(_im2).set_label("dZ/dy")
_ax.set_xlabel("x")
_ax.set_ylabel("y")

_ax = _axs[2]
_im3 = _ax.imshow(
    np.diff(Z,axis=1),
    aspect='auto',
    cmap="RdBu_r", 
    origin='lower',
    extent=(x_4[0], x_4[-1], y_3[0], y_3[-1])
)
_fig.colorbar(_im3).set_label("dZ/dx")
_ax.set_xlabel("x")
_ax.set_ylabel("y")

_fig.tight_layout()
show(_fig)